# Notebook 3: RAG and AI Reasoning Pipeline
## AlphaForge AI — Explainable Financial Forecasting with Multi-Agent Intelligence

**Purpose**

This notebook:

1. Collects financial news articles for the target ticker.
2. Builds a **FAISS** vector database from news embeddings (RAG knowledge base).
3. Implements the **three AI agents**:
   - Agent 1: Quantitative Analyst (reads forecast summary from Notebook 2)
   - Agent 2: News Intelligence Agent (retrieves relevant news, computes sentiment)
   - Agent 3: Investment Advisor (generates BUY/HOLD/SELL recommendation with explanation)
4. Produces an **explainable final output** including confidence, reasoning, and supporting evidence.
5. (Optional) Interactive **Financial RAG Chat Assistant**.

All components are fully integrated with the outputs of Notebook 1 and Notebook 2.

In [ ]:
# ============================================================
# 0. Install / Import Dependencies
# ============================================================
!pip install yfinance faiss-cpu sentence-transformers transformers --quiet

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

import faiss
from sentence_transformers import SentenceTransformer

# Optional: FinBERT for sentiment, small LLM for explanation
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

print("Libraries imported.")

In [ ]:
# ============================================================
# 1. Configuration (must match Notebook 1 & 2)
# ============================================================
TICKER = "AAPL"                     # Same as Notebook 1
DATA_DIR = "data"
RESULTS_DIR = "results"
FORECAST_SUMMARY_PATH = os.path.join(RESULTS_DIR, "forecast_summary.json")
PROCESSED_DATA_PATH = os.path.join(DATA_DIR, "processed_features.csv")

# RAG parameters
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
FAISS_INDEX_PATH = os.path.join(RESULTS_DIR, "news_index.faiss")
DOCS_META_PATH = os.path.join(RESULTS_DIR, "news_metadata.json")
TOP_K = 5                           # number of documents to retrieve
USE_LLM = False                     # set to True to use a local LLM for explanation
# (requires additional model download ~1GB)

print(f"Ticker: {TICKER}")
print(f"Forecast summary expected at: {FORECAST_SUMMARY_PATH}")

## 2. Load Forecast Summary from Notebook 2

This is the quantitative foundation produced by Agent 1.

In [ ]:
# ============================================================
# 2. Load Forecast Summary
# ============================================================
with open(FORECAST_SUMMARY_PATH, "r") as f:
    forecast = json.load(f)

print("Forecast Summary:")
print(json.dumps(forecast, indent=2))

## 3. Build Financial News Knowledge Base (RAG Pipeline)

We fetch recent news for the ticker using `yfinance`, embed the titles, and create a FAISS index.

In [ ]:
# ============================================================
# 3.1 Fetch News via yfinance
# ============================================================
ticker_obj = yf.Ticker(TICKER)
news_list = ticker_obj.news  # returns list of dicts
print(f"Number of news articles retrieved: {len(news_list)}")

# Extract relevant fields: title, link, publisher, timestamp
documents = []
for item in news_list:
    title = item.get("title", "")
    link = item.get("link", "")
    publisher = item.get("publisher", "")
    pub_time = item.get("providerPublishTime", None)
    if title:
        documents.append({
            "title": title,
            "link": link,
            "publisher": publisher,
            "timestamp": pub_time,
            "text": title           # we embed the title (can be extended)
        })

print(f"Documents created: {len(documents)}")
if documents:
    print("Example document:")
    print(documents[0])

In [ ]:
# ============================================================
# 3.2 Generate Embeddings
# ============================================================
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

doc_texts = [doc["text"] for doc in documents]
doc_embeddings = embedder.encode(doc_texts, show_progress_bar=True)
print("Embeddings shape:", doc_embeddings.shape)

In [ ]:
# ============================================================
# 3.3 Create FAISS Index
# ============================================================
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)   # L2 distance
index.add(doc_embeddings.astype(np.float32))
print(f"FAISS index contains {index.ntotal} vectors.")

# Save index and metadata
faiss.write_index(index, FAISS_INDEX_PATH)
with open(DOCS_META_PATH, "w") as f:
    json.dump(documents, f, indent=2)
print(f"Index saved to {FAISS_INDEX_PATH}")
print(f"Metadata saved to {DOCS_META_PATH}")

## 4. Define the Three AI Agents

In [ ]:
# ============================================================
# Agent 1: Quantitative Analyst
# ============================================================
def agent_quantitative_analyst(forecast):
    """
    Reads the forecast summary and returns:
      - primary model name
      - last close, predicted close, predicted change %
      - direction and model confidence (directional accuracy)
      - volatility, RSI, additional metrics
    """
    model_name = forecast["primary_model"]
    metrics = forecast["model_metrics"][model_name]
    dir_acc = metrics["Directional_Accuracy"]

    analysis = {
        "as_of_date": forecast["as_of_date"],
        "last_close": forecast["last_close"],
        "predicted_next_close": forecast["predicted_next_close"],
        "predicted_change_pct": forecast["predicted_change_pct"],
        "predicted_direction": forecast["predicted_direction"],
        "model_confidence": dir_acc,           # raw directional accuracy as confidence base
        "volatility_20d": forecast["recent_volatility_20d"],
        "rsi_14": forecast["recent_rsi_14"],
        "sharpe": metrics["Sharpe_Ratio"]
    }
    return analysis

qa_result = agent_quantitative_analyst(forecast)
print("Quantitative Analysis:")
print(json.dumps(qa_result, indent=2))

In [ ]:
# ============================================================
# Agent 2: News Intelligence Agent (RAG retrieval + sentiment)
# ============================================================
def retrieve_relevant_news(query, top_k=TOP_K):
    """Embed query, search FAISS, return top-k documents."""
    query_vec = embedder.encode([query])
    distances, indices = index.search(query_vec.astype(np.float32), top_k)
    results = []
    for idx in indices[0]:
        if idx != -1:
            results.append(documents[idx])
    return results

def analyze_sentiment(text):
    """Use FinBERT to get sentiment label and score."""
    # Initialize FinBERT only once
    if "finbert_pipeline" not in globals():
        global finbert_pipeline
        finbert_pipeline = pipeline("text-classification",
                                    model="ProsusAI/finbert",
                                    return_all_scores=True)
    scores = finbert_pipeline(text)
    # scores is list of dicts: [{'label': 'positive', 'score': 0.7}, ...]
    return scores[0]  # list of three labels

def agent_news_intelligence(quant_analysis, ticker=TICKER):
    """
    Construct a query based on market state, retrieve news,
    measure sentiment, and return evidence.
    """
    direction = quant_analysis["predicted_direction"]
    vol = quant_analysis["volatility_20d"]
    rsi = quant_analysis["rsi_14"]

    # Create a descriptive query string
    query = f"{ticker} stock price {'rise' if direction=='UP' else 'fall'} forecast market news sentiment volatility RSI {rsi}"
    print(f"Query: '{query}'")

    retrieved = retrieve_relevant_news(query, top_k=TOP_K)
    evidence = []
    sentiments = []
    for doc in retrieved:
        sentiment_result = analyze_sentiment(doc["text"])
        # sentiment_result is list of {label, score} for positive, negative, neutral
        sentiment_dict = {s["label"]: s["score"] for s in sentiment_result}
        sentiments.append(sentiment_dict)
        evidence.append({
            "title": doc["title"],
            "link": doc["link"],
            "publisher": doc["publisher"],
            "sentiment": sentiment_dict
        })

    # Aggregate sentiment
    agg_sentiment = {"positive": 0, "negative": 0, "neutral": 0}
    for s in sentiments:
        for k in agg_sentiment:
            agg_sentiment[k] += s.get(k, 0)
    # Normalize (average)
    n = len(sentiments)
    if n > 0:
        for k in agg_sentiment:
            agg_sentiment[k] /= n

    return {
        "query": query,
        "retrieved_articles": evidence,
        "aggregate_sentiment": agg_sentiment
    }

news_intel = agent_news_intelligence(qa_result, TICKER)
print("News Intelligence:")
print(f"Aggregated sentiment: {news_intel['aggregate_sentiment']}")
print(f"Retrieved {len(news_intel['retrieved_articles'])} articles.")

In [ ]:
# ============================================================
# Agent 3: Investment Advisor
# ============================================================
def generate_recommendation(quant, news_intel, use_llm=False):
    """
    Combine quantitative and news insights to produce BUY / HOLD / SELL.
    Confidence = (model_dir_acc + sentiment_agreement_weight) / 2
    """
    direction = quant["predicted_direction"]
    model_conf = quant["model_confidence"] / 100.0  # normalize to 0-1
    sent = news_intel["aggregate_sentiment"]

    # Determine sentiment signal: 1 if positive > negative, -1 if negative > positive, else 0
    if sent["positive"] > sent["negative"]:
        sent_signal = 1
        sent_conf = sent["positive"]
    elif sent["negative"] > sent["positive"]:
        sent_signal = -1
        sent_conf = sent["negative"]
    else:
        sent_signal = 0
        sent_conf = sent["neutral"]

    # Map direction to signal: UP=1, DOWN=-1
    dir_signal = 1 if direction == "UP" else -1

    # Rule-based recommendation
    if dir_signal == 1 and sent_signal == 1:
        recommendation = "BUY"
    elif dir_signal == -1 and sent_signal == -1:
        recommendation = "SELL"
    elif dir_signal == 1 and sent_signal == 0:
        recommendation = "HOLD"
    elif dir_signal == -1 and sent_signal == 0:
        recommendation = "HOLD"
    else:
        recommendation = "HOLD"   # mixed signals

    # Confidence score (simple average of model and sentiment confidence)
    confidence = (model_conf + sent_conf) / 2 * 100

    # Build explanation (template-based)
    reasons = []
    if direction == "UP":
        reasons.append(f"Quantitative model predicts a {quant['predicted_change_pct']:.2f}% increase.")
    else:
        reasons.append(f"Quantitative model predicts a {quant['predicted_change_pct']:.2f}% decrease.")

    if sent_signal == 1:
        reasons.append("News sentiment is predominantly positive.")
    elif sent_signal == -1:
        reasons.append("News sentiment is predominantly negative.")
    else:
        reasons.append("News sentiment is neutral/mixed.")

    # Add volatility context
    if quant["volatility_20d"] > 0.02:
        reasons.append("High recent volatility indicates elevated risk.")
    if quant["rsi_14"] > 70:
        reasons.append("RSI indicates overbought conditions.")
    elif quant["rsi_14"] < 30:
        reasons.append("RSI indicates oversold conditions.")

    # Generate natural language explanation using LLM (optional)
    if use_llm:
        # Load a small LLM (e.g., FLAN-T5) for reasoning
        if "llm_tokenizer" not in globals():
            global llm_tokenizer, llm_model
            llm_name = "google/flan-t5-base"
            llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
            llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

        prompt = f"""Ticker: {TICKER}
Date: {quant['as_of_date']}
Last close: {quant['last_close']:.2f}
Forecast: {direction} {quant['predicted_change_pct']:.2f}%
Model confidence: {model_conf*100:.0f}%
Sentiment: positive {sent['positive']:.2f}, negative {sent['negative']:.2f}
Key news headlines: {', '.join([a['title'] for a in news_intel['retrieved_articles'][:3]])}
Explain the investment recommendation and the main factors behind it."""
        inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        outputs = llm_model.generate(**inputs, max_length=150)
        explanation_text = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
        reasons.append(f"LLM insight: {explanation_text}")

    explanation = " ".join(reasons)

    return {
        "recommendation": recommendation,
        "confidence": round(confidence, 1),
        "quantitative_forecast": direction,
        "sentiment_signal": "positive" if sent_signal==1 else ("negative" if sent_signal==-1 else "neutral"),
        "explanation": explanation,
        "supporting_evidence": news_intel["retrieved_articles"]
    }

# Generate final recommendation (without LLM by default)
final_rec = generate_recommendation(qa_result, news_intel, use_llm=USE_LLM)
print("=== FINAL INVESTMENT RECOMMENDATION ===")
print(f"Recommendation: {final_rec['recommendation']}")
print(f"Confidence: {final_rec['confidence']}%")
print(f"Explanation: {final_rec['explanation']}")
print(f"Supporting evidence ({len(final_rec['supporting_evidence'])} articles):")
for i, doc in enumerate(final_rec['supporting_evidence']):
    print(f"  [{i+1}] {doc['title']}")

## 5. Optional Extension: Financial RAG Chat Assistant

Answer questions about the prediction using the knowledge base.

In [ ]:
# ============================================================
# Interactive Q&A (simple) — run this cell to ask questions
# ============================================================
def chat_assistant(question):
    """Retrieve relevant documents and provide a brief answer."""
    retrieved = retrieve_relevant_news(question, top_k=3)
    # Also include current forecast context
    context = f"Current {TICKER} price is {qa_result['last_close']:.2f}, predicted direction {qa_result['predicted_direction']}."
    titles = [d['title'] for d in retrieved]
    answer = f"Based on recent news: {'; '.join(titles)}. {context}"
    return answer

# Example questions
example_questions = [
    f"Why is the model recommending {final_rec['recommendation']}?",
    "What news influenced the prediction?",
    "What are the biggest risks?"
]

print("Example Q&A:")
for q in example_questions:
    print(f"Q: {q}")
    print(f"A: {chat_assistant(q)}")
    print("-"*80)

## 6. Summary

- Built a FAISS-based RAG pipeline from financial news titles.
- Implemented Agent 1 (Quantitative Analyst) using Notebook 2's forecast summary.
- Implemented Agent 2 (News Intelligence) with retrieval and FinBERT sentiment.
- Implemented Agent 3 (Investment Advisor) producing an explainable recommendation.
- The final output includes confidence, reasons, and supporting evidence.
- Optionally demonstrated a chat assistant for interactive queries.

This completes the **AlphaForge AI** explainable financial forecasting framework.